In [ ]:
# S4S-0  Setup (Weather-File Stress Test, 1A + 7)
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, shutil, subprocess, sys, platform, time
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import numpy as np
import pandas as pd

BASE        = '/content/drive/MyDrive/BEM-LLM'
DATA        = f'{BASE}/data'
REF         = f'{DATA}/idf_referans'
WEATHER_DIR = f'{DATA}/epw_iklim'
ANALIZ      = f'{BASE}/analiz'
SPRINT2_DIR = f'{ANALIZ}/sprint2_lobo'
PROMPTS_DIR = f'{ANALIZ}/sprint3_prompts'
SIM_DIR     = f'{ANALIZ}/sprint4_simulation'
IDF_OUT     = f'{BASE}/idf_llm'
EP_OUT      = f'{BASE}/sonuclar'
os.makedirs(IDF_OUT, exist_ok=True)
os.makedirs(EP_OUT, exist_ok=True)

EP = '/usr/local/EnergyPlus/energyplus'
EP_TIMEOUT_S = 1800
EP_VERSION_TAG = '22.1.0-ed759b17ee'
EP_DOWNLOAD_URL = ('https://github.com/NREL/EnergyPlus/releases/download/v22.1.0/'
                    'EnergyPlus-22.1.0-ed759b17ee-Linux-Ubuntu20.04-x86_64.tar.gz')

installed_version = None
if os.path.exists(EP):
    installed_version = subprocess.run([EP, '--version'], capture_output=True, text=True).stdout.strip()

if not os.path.exists(EP) or EP_VERSION_TAG not in (installed_version or ''):
    if os.path.exists('/usr/local/EnergyPlus'):
        subprocess.run(['rm', '-rf', '/usr/local/EnergyPlus'])
    subprocess.run(['wget', '-q', '-O', '/tmp/ep.tar.gz', EP_DOWNLOAD_URL])
    os.makedirs('/usr/local/EnergyPlus', exist_ok=True)
    subprocess.run(['tar', '-xzf', '/tmp/ep.tar.gz', '-C', '/usr/local/EnergyPlus', '--strip-components=1'])

energyplus_version = subprocess.run([EP, '--version'], capture_output=True, text=True).stdout.strip()
print(energyplus_version)
assert '22.1' in energyplus_version, f'Expected EnergyPlus 22.1.x, got: {energyplus_version}'

with open(f'{ANALIZ}/protocol.json') as f:
    protocol = json.load(f)
with open(f'{SPRINT2_DIR}/lobo_range_table.json') as f:
    lobo = json.load(f)
with open(f'{SPRINT2_DIR}/control_parameters.json') as f:
    control_data = json.load(f)

TESTED_BUILDINGS = protocol['tested_buildings']
MODELS   = protocol['models']
SEEDS    = protocol['seeds']
CLIMATES = protocol['climates']
TARGET_PARAMS  = lobo['target_parameters']
CONTROL_PARAMS = lobo['control_parameters']
CONTROL_VALUES = control_data['values_by_building']

CLIMATE_EPW = {
    '1A': f'{WEATHER_DIR}/1A_Miami.epw',
    '7':  f'{WEATHER_DIR}/7_InternationalFalls.epw',
}

TOLERANCE_PCT = {
    'window_u_value': 0.5, 'window_shgc': 0.5, 'lighting_W_m2': 0.5,
    'equipment_W_m2': 1.0, 'occupancy_m2_person': 1.0, 'outdoor_air': 2.0,
    'cooling_cop': 1.0, 'heating_setpoint_C': 0.5, 'cooling_setpoint_C': 0.5,
}

N_WORKERS = multiprocessing.cpu_count()
print(f'{len(TESTED_BUILDINGS)} buildings, {len(MODELS)} models, {len(SEEDS)} seeds, '
      f'{multiprocessing.cpu_count()} CPUs ({N_WORKERS} workers)')

Mounted at /content/drive
EnergyPlus, Version 22.1.0-ed759b17ee, YMD=2026.07.28 16:58
5 buildings, 2 models, 5 seeds, 2 CPUs (2 workers)


In [ ]:
# S4S-A  Comment-anchored parsing (identical logic to S4M-A)
def load_idf_text(path):
    with open(path, 'r', errors='ignore') as f:
        return f.read()


def safe_float(v):
    try:
        f = float(v)
        return f if np.isfinite(f) else None
    except (TypeError, ValueError):
        return None


def weighted_mean(values, weights):
    values, weights = np.array(values, dtype=float), np.array(weights, dtype=float)
    return float(np.average(values, weights=weights)) if len(values) else None


def iter_blocks(idf_text):
    parts = re.split(r'(;[ \t]*(?:!-[^\n]*)?\n)', idf_text)
    blocks = [''.join(parts[i:i + 2]) for i in range(0, len(parts) - 1, 2)]
    if len(parts) % 2 == 1:
        blocks.append(parts[-1])
    for block in blocks:
        stripped = block.strip()
        if not stripped:
            continue
        obj_type = None
        for line in stripped.split('\n'):
            line_s = line.strip()
            if not line_s or line_s.startswith('!'):
                continue
            obj_type = line_s.split(',')[0].strip().upper()
            break
        if obj_type:
            yield obj_type, block


def blocks_of(idf_text, object_type):
    return [b for t, b in iter_blocks(idf_text) if t == object_type.upper()]


def value_before_comment(block, comment_substring):
    m = re.search(rf'([\w.+\-]+)\s*[,;]\s*!-[^\n]*{re.escape(comment_substring)}', block, re.IGNORECASE)
    return safe_float(m.group(1)) if m else None


def text_before_comment(block, comment_substring):
    m = re.search(rf'([^\n,;]+)\s*[,;]\s*!-[^\n]*{re.escape(comment_substring)}', block, re.IGNORECASE)
    return m.group(1).strip() if m else None


def polygon_area(block):
    coords = re.findall(r'([\-\d.]+)\s*,\s*([\-\d.]+)\s*,\s*([\-\d.]+)\s*[,;]?\s*!-[^\n]*Vertex', block)
    if len(coords) < 3:
        return None
    pts = [np.array([float(x), float(y), float(z)]) for x, y, z in coords]
    normal = np.zeros(3)
    for i in range(len(pts)):
        normal += np.cross(pts[i], pts[(i + 1) % len(pts)])
    return float(np.linalg.norm(normal)) / 2.0


def zone_area_map(idf_text):
    areas = {}
    for block in blocks_of(idf_text, 'BuildingSurface:Detailed'):
        if (text_before_comment(block, 'Surface Type') or '').strip().lower() != 'floor':
            continue
        zone = text_before_comment(block, 'Zone Name')
        area = polygon_area(block)
        if zone and area:
            areas[zone.strip().upper()] = areas.get(zone.strip().upper(), 0.0) + area
    return areas


def get_window_u_shgc(idf_text):
    con_glazing = {}
    for block in blocks_of(idf_text, 'Construction'):
        name = text_before_comment(block, 'Name')
        outside_layer = text_before_comment(block, 'Outside Layer')
        if name and outside_layer:
            con_glazing[name.upper()] = outside_layer.rstrip(';').upper()

    glazing = {}
    for block in blocks_of(idf_text, 'WindowMaterial:SimpleGlazingSystem'):
        name = text_before_comment(block, 'Name')
        u = value_before_comment(block, 'U-Factor')
        shgc = value_before_comment(block, 'Solar Heat Gain Coefficient')
        if name and u and shgc:
            glazing[name.upper()] = (u, shgc)

    us, shgcs, areas = [], [], []
    for block in blocks_of(idf_text, 'FenestrationSurface:Detailed'):
        if (text_before_comment(block, 'Surface Type') or '').strip().lower() != 'window':
            continue
        con_name = text_before_comment(block, 'Construction Name')
        props = glazing.get(con_glazing.get((con_name or '').upper(), '').upper())
        if not props:
            continue
        area = polygon_area(block) or 1.0
        us.append(props[0]); shgcs.append(props[1]); areas.append(area)
    if not us:
        return None, None
    return round(weighted_mean(us, areas), 4), round(weighted_mean(shgcs, areas), 4)


def get_lighting(idf_text, zmap):
    vals, wts = [], []
    for block in blocks_of(idf_text, 'Lights'):
        v = value_before_comment(block, 'Watts per Zone Floor Area')
        if v:
            zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
            vals.append(v); wts.append(zmap.get(zone, 1.0))
    return round(weighted_mean(vals, wts), 3) if vals else None


EXCLUDE_ZONES = ['CORRIDOR', 'MECH', 'BATH', 'LOBBY', 'STAIR', 'PLENUM', 'JANITOR']


def get_equipment(idf_text, zmap, building_name=None):
    exclude = EXCLUDE_ZONES + (['STORAGE'] if building_name != 'Warehouse' else [])
    vals, wts = [], []
    for block in blocks_of(idf_text, 'ElectricEquipment'):
        zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
        if any(kw in zone for kw in exclude):
            continue
        area = zmap.get(zone)
        wa = value_before_comment(block, 'Watts per Zone Floor Area')
        if wa:
            vals.append(wa); wts.append(area or 1.0)
            continue
        design_w = value_before_comment(block, 'Design Level {W}')
        if design_w and area:
            density = design_w / area
            if density <= 150.0:
                vals.append(density); wts.append(area)
    return round(weighted_mean(vals, wts), 3) if vals else None


def get_occupancy(idf_text, zmap):
    vals, wts = [], []
    for block in blocks_of(idf_text, 'People'):
        zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
        area = zmap.get(zone)
        apf = value_before_comment(block, 'Floor Area per Person')
        if apf:
            vals.append(apf); wts.append(area or 1.0)
            continue
        ppa = value_before_comment(block, 'People per Floor Area')
        if ppa:
            vals.append(1.0 / ppa); wts.append(area or 1.0)
            continue
        m = re.search(r'([\w.+\-]+)\s*,\s*!-\s*Number of People\s*\n', block, re.IGNORECASE)
        n = safe_float(m.group(1)) if m else None
        if n and area:
            vals.append(area / n); wts.append(area)
    return round(weighted_mean(vals, wts), 3) if vals else None


def get_outdoor_air(idf_text, occupancy_density):
    vals, wts = [], []
    for block in blocks_of(idf_text, 'DesignSpecification:OutdoorAir'):
        method = (text_before_comment(block, 'Outdoor Air Method') or '').strip().lower()
        per_area = value_before_comment(block, 'Outdoor Air Flow per Zone Floor Area') or 0.0
        per_person_area = (value_before_comment(block, 'Outdoor Air Flow per Person') or 0.0) / (occupancy_density or 18.0)
        combined = max(per_area, per_person_area) if method == 'maximum' else per_area + per_person_area
        if combined > 0:
            vals.append(combined); wts.append(1.0)
    return round(weighted_mean(vals, wts), 6) if vals else None


def get_cooling_cop(idf_text):
    vals, wts = [], []
    for obj_type in ['Coil:Cooling:DX:SingleSpeed', 'Coil:Cooling:DX:TwoSpeed', 'Coil:Cooling:DX:MultiSpeed']:
        for block in blocks_of(idf_text, obj_type):
            caps = [safe_float(v) for v in re.findall(r'([\d.]+)\s*,\s*!-[^\n]*Rated Total Cooling Capacity', block)]
            cops = [safe_float(v) for v in re.findall(r'([\d.]+)\s*,\s*!-[^\n]*Rated Cooling COP', block)]
            caps = [c for c in caps if c is not None]
            cops = [c for c in cops if c is not None]
            for i, cop in enumerate(cops):
                vals.append(cop); wts.append(caps[i] if i < len(caps) else 1.0)
    return round(weighted_mean(vals, wts), 4) if vals else None


def get_setpoints(idf_text):
    sched_vals = {}
    for block in blocks_of(idf_text, 'Schedule:Compact'):
        name = text_before_comment(block, 'Name')
        vals = [safe_float(v) for v in re.findall(r'Until:\s*\d{1,2}:\d{2}\s*,\s*([\d.\-]+)', block)]
        vals = [v for v in vals if v is not None]
        if name and vals:
            sched_vals[name.upper()] = vals

    heat_vals, cool_vals = [], []
    for block in blocks_of(idf_text, 'ThermostatSetpoint:DualSetpoint'):
        heat_sched = text_before_comment(block, 'Heating Setpoint Temperature Schedule Name')
        cool_sched = text_before_comment(block, 'Cooling Setpoint Temperature Schedule Name')
        h = [v for v in sched_vals.get((heat_sched or '').upper(), []) if 18.0 <= v <= 26.0]
        c = [v for v in sched_vals.get((cool_sched or '').upper(), []) if 20.0 <= v <= 27.0]
        if h:
            heat_vals.append(float(np.median(h)))
        if c:
            cool_vals.append(float(np.median(c)))
    heat_sp = round(float(np.mean(heat_vals)), 2) if heat_vals else None
    cool_sp = round(float(np.mean(cool_vals)), 2) if cool_vals else None
    return heat_sp, cool_sp


def extract_all_params(idf_text, building_name=None):
    zmap = zone_area_map(idf_text)
    u, shgc = get_window_u_shgc(idf_text)
    occ = get_occupancy(idf_text, zmap)
    heat_sp, cool_sp = get_setpoints(idf_text)
    return {
        'window_u_value': u, 'window_shgc': shgc,
        'lighting_W_m2': get_lighting(idf_text, zmap),
        'equipment_W_m2': get_equipment(idf_text, zmap, building_name),
        'occupancy_m2_person': occ,
        'outdoor_air': get_outdoor_air(idf_text, occ),
        'cooling_cop': get_cooling_cop(idf_text),
        'heating_setpoint_C': heat_sp, 'cooling_setpoint_C': cool_sp,
    }

In [ ]:
# S4S-B  Injection (identical logic to S4M-B)
def _replace_before_comment(block, comment_substring, new_value):
    pattern = re.compile(rf'([\w.+\-]*)(\s*[,;]\s*!-[^\n]*{re.escape(comment_substring)})', re.IGNORECASE)
    return pattern.sub(lambda m: f'{new_value}{m.group(2)}', block, count=1)


def _rebuild(idf_text, object_type, transform):
    out = []
    for otype, block in iter_blocks(idf_text):
        out.append(transform(block) if otype == object_type.upper() else block)
    return ''.join(out)


def inject_window_u_shgc(idf_text, u_value, shgc):
    def fix(block):
        block = _replace_before_comment(block, 'U-Factor', round(u_value, 4))
        return _replace_before_comment(block, 'Solar Heat Gain Coefficient', round(shgc, 4))
    return _rebuild(idf_text, 'WindowMaterial:SimpleGlazingSystem', fix)


def inject_lights(idf_text, value_w_m2):
    def fix(block):
        if value_before_comment(block, 'Watts per Zone Floor Area') is not None:
            return _replace_before_comment(block, 'Watts per Zone Floor Area', round(value_w_m2, 3))
        return block
    return _rebuild(idf_text, 'Lights', fix)


def inject_equipment(idf_text, value_w_m2, zmap):
    def fix(block):
        if value_before_comment(block, 'Watts per Zone Floor Area') is not None:
            return _replace_before_comment(block, 'Watts per Zone Floor Area', round(value_w_m2, 3))
        if value_before_comment(block, 'Design Level {W}') is not None:
            zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
            area = zmap.get(zone)
            if area:
                return _replace_before_comment(block, 'Design Level {W}', round(value_w_m2 * area, 2))
        return block
    return _rebuild(idf_text, 'ElectricEquipment', fix)


def inject_occupancy(idf_text, value_m2_person, zmap):
    def fix(block):
        if value_before_comment(block, 'Floor Area per Person') is not None:
            return _replace_before_comment(block, 'Floor Area per Person', round(value_m2_person, 3))
        if value_before_comment(block, 'People per Floor Area') is not None:
            return _replace_before_comment(block, 'People per Floor Area', round(1.0 / value_m2_person, 6))
        m = re.search(r'([\w.+\-]+)(\s*,\s*!-\s*Number of People\s*\n)', block, re.IGNORECASE)
        if m and safe_float(m.group(1)) is not None:
            zone = (text_before_comment(block, 'Zone or ZoneList') or '').strip().upper()
            area = zmap.get(zone)
            if area:
                new_n = round(area / value_m2_person, 3)
                return block[:m.start(1)] + str(new_n) + block[m.end(1):]
        return block
    return _rebuild(idf_text, 'People', fix)


def inject_outdoor_air(idf_text, value_m3_s_m2):
    def fix(block):
        return _replace_before_comment(block, 'Outdoor Air Flow per Zone Floor Area', round(value_m3_s_m2, 6))
    return _rebuild(idf_text, 'DesignSpecification:OutdoorAir', fix)


def inject_cooling_cop(idf_text, cop):
    pattern = re.compile(r'([\d.]+)(\s*,\s*!-[^\n]*Rated Cooling COP)', re.IGNORECASE)
    return pattern.sub(lambda m: f'{round(cop, 3)}{m.group(2)}', idf_text)


def inject_setpoint_schedule(idf_text, schedule_name, value_c):
    block_pattern = re.compile(rf'(Schedule:Compact\s*,\s*{re.escape(schedule_name)}\s*,.*?;)',
                               re.IGNORECASE | re.DOTALL)
    match = block_pattern.search(idf_text)
    if not match:
        return idf_text
    new_block = re.sub(r'(Until:\s*\d{1,2}:\d{2}\s*,)\s*[\d.\-]+', rf'\g<1>{round(value_c, 2)}', match.group(1))
    return idf_text.replace(match.group(1), new_block)


def inject_setpoints(idf_text, heating_c, cooling_c):
    for block in blocks_of(idf_text, 'ThermostatSetpoint:DualSetpoint'):
        heat_sched = text_before_comment(block, 'Heating Setpoint Temperature Schedule Name')
        cool_sched = text_before_comment(block, 'Cooling Setpoint Temperature Schedule Name')
        if heat_sched:
            idf_text = inject_setpoint_schedule(idf_text, heat_sched, heating_c)
        if cool_sched:
            idf_text = inject_setpoint_schedule(idf_text, cool_sched, cooling_c)
    return idf_text


def inject_all_parameters(idf_path, out_path, values):
    text = load_idf_text(idf_path)
    zmap = zone_area_map(text)
    text = inject_window_u_shgc(text, values['window_u_value'], values['window_shgc'])
    text = inject_lights(text, values['lighting_W_m2'])
    text = inject_equipment(text, values['equipment_W_m2'], zmap)
    text = inject_occupancy(text, values['occupancy_m2_person'], zmap)
    text = inject_outdoor_air(text, values['outdoor_air'])
    text = inject_cooling_cop(text, values['cooling_cop'])
    text = inject_setpoints(text, values['heating_setpoint_C'], values['cooling_setpoint_C'])
    with open(out_path, 'w') as f:
        f.write(text)

In [ ]:
# S4S-C  Ensures identical output reporting across every generated IDF
STANDARD_OUTPUTS = """
Output:Meter,Electricity:Facility,Monthly;
Output:Meter,NaturalGas:Facility,Monthly;
Output:Meter,DistrictHeating:Facility,Monthly;
Output:Meter,DistrictCooling:Facility,Monthly;
Output:Variable,*,Zone Air System Sensible Heating Energy,Monthly;
Output:Variable,*,Zone Air System Sensible Cooling Energy,Monthly;
"""

def _strip_idf_comments(idf_text):
    return re.sub(r'!.*', '', idf_text)

def ensure_standard_outputs(idf_path):
    text = load_idf_text(idf_path)
    active_text = _strip_idf_comments(text)
    if 'Output:Meter,Electricity:Facility' not in active_text:
        with open(idf_path, 'w') as f:
            f.write(text.rstrip() + '\n' + STANDARD_OUTPUTS)

In [ ]:
# S4S-D  Round-trip verification + EnergyPlus runner
def round_trip_check(idf_path, target_values, building_name=None):
    readback = extract_all_params(load_idf_text(idf_path), building_name)
    report, passed = [], True
    for param, target in target_values.items():
        actual = readback.get(param)
        tol = TOLERANCE_PCT.get(param, 0.5)
        if actual is None:
            report.append({'parameter': param, 'target': target, 'readback': None, 'status': 'MISSING'})
            passed = False
            continue
        diff_pct = abs(actual - target) / abs(target) * 100 if target else abs(actual - target) * 100
        ok = diff_pct <= tol
        passed = passed and ok
        report.append({'parameter': param, 'target': round(target, 6), 'readback': round(actual, 6),
                        'diff_pct': round(diff_pct, 4), 'tolerance_pct': tol,
                        'status': 'OK' if ok else 'TOLERANCE_EXCEEDED'})
    return passed, report


def build_and_check(building, values, tag):
    out_idf = f'{IDF_OUT}/{tag}.idf'
    shutil.copy(f'{REF}/ASHRAE901_{building}_STD2019_Buffalo.idf', out_idf)
    inject_all_parameters(out_idf, out_idf, values)
    ensure_standard_outputs(out_idf)
    return out_idf, round_trip_check(out_idf, values, building_name=building)


def run_energyplus(idf_path, epw_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    try:
        result = subprocess.run([EP, '-w', epw_path, '-d', out_dir, '-r', idf_path],
                                capture_output=True, text=True, timeout=EP_TIMEOUT_S)
        return result.returncode == 0, result.stderr
    except subprocess.TimeoutExpired:
        return False, f'TIMEOUT: exceeded {EP_TIMEOUT_S}s'

In [ ]:
# S4S-E  Weather-file stress-test manifest: 100 F4-only LLM outputs
# (5 buildings x 2 models x 5 seeds x 2 climates) + 20 deterministic
# baselines (5 buildings x 2 baseline types x 2 climates) = 120 vectors.
# No DOE reference, no uniform-random.
param_debug = pd.read_csv(f'{PROMPTS_DIR}/param_debug_mapping.csv')
ep_baselines = pd.read_csv(f'{SPRINT2_DIR}/energyplus_baseline_vectors.csv')

stress_vectors = []
llm_stress = param_debug[(param_debug['format'] == 'F4') & (param_debug['climate'] != CLIMATES['main'])]
for (building, climate, model, seed), group in llm_stress.groupby(['building', 'climate', 'model', 'seed']):
    values = dict(zip(group['true_name'], group['value']))
    if set(TARGET_PARAMS).issubset(values):
        stress_vectors.append({'building': building, 'climate': climate,
                                'source': f'F4_{model.split("/")[-1]}_seed{seed}', 'source_type': 'llm',
                                'format': 'F4', 'model': model, 'seed': seed,
                                **{p: values[p] for p in TARGET_PARAMS}})

for climate in CLIMATES['validation']:
    for _, row in ep_baselines.iterrows():
        stress_vectors.append({'building': row['building'], 'climate': climate,
                                'source': row['baseline_type'], 'source_type': 'deterministic_baseline',
                                'format': None, 'model': None, 'seed': None,
                                **{p: row[p] for p in TARGET_PARAMS}})

stress_manifest_df = pd.DataFrame(stress_vectors)
stress_manifest_df.to_csv(f'{SIM_DIR}/simulation_vector_manifest_weather_stress.csv', index=False)
print(f'{len(stress_manifest_df)} weather-file stress-test vectors registered.')
print(stress_manifest_df.groupby(['climate', 'source_type']).size().to_string())

120 weather-file stress-test vectors registered.
climate  source_type           
1A       deterministic_baseline    10
         llm                       50
7        deterministic_baseline    10
         llm                       50


In [ ]:
# S4S-F  Smoke test: one per (model, climate) for F4, one per (baseline, climate)
def smoke_test_vector(row, epw_path, index, total):
    tag = f'{row["building"]}_{row["climate"]}_{row["source"]}'
    print(f'[{index}/{total}] {tag} ...', end=' ', flush=True)
    t0 = time.time()
    values = {p: row[p] for p in TARGET_PARAMS}
    out_idf, (rt_passed, rt_report) = build_and_check(row['building'], values, tag)
    if not rt_passed:
        print(f'ROUND-TRIP FAILED ({time.time() - t0:.1f}s)')
        return {'tag': tag, 'round_trip_ok': False, 'energyplus_ok': None, 'report': rt_report}
    ep_ok, stderr = run_energyplus(out_idf, epw_path, f'{EP_OUT}/{tag}')
    status = 'OK' if ep_ok else ('TIMEOUT' if 'TIMEOUT' in str(stderr) else 'EP FAILED')
    print(f'{status} ({time.time() - t0:.1f}s)')
    return {'tag': tag, 'round_trip_ok': True, 'energyplus_ok': ep_ok,
            'stderr_tail': stderr[-500:] if not ep_ok else None}

per_format_model_climate = stress_manifest_df[stress_manifest_df.source_type == 'llm'].groupby(['model', 'climate']).head(1)
per_baseline_climate = stress_manifest_df[stress_manifest_df.source_type == 'deterministic_baseline'].groupby(['source', 'climate']).head(1)
smoke_sample = pd.concat([per_format_model_climate, per_baseline_climate], ignore_index=True)
print(f'Smoke sample: {len(smoke_sample)} cases\n')

batch_start = time.time()
smoke_results = []
for i, (_, row) in enumerate(smoke_sample.iterrows(), start=1):
    smoke_results.append(smoke_test_vector(row, CLIMATE_EPW[row['climate']], i, len(smoke_sample)))
    pd.DataFrame(smoke_results).to_csv(f'{SIM_DIR}/smoke_test_results_stress.csv', index=False)
    elapsed = time.time() - batch_start
    print(f'    -> avg {elapsed/i:.1f}s/case, est. remaining {elapsed/i*(len(smoke_sample)-i)/60:.1f} min')

smoke_df = pd.DataFrame(smoke_results)
rt_rejected = (~smoke_df['round_trip_ok']).sum()
ep_eligible = smoke_df[smoke_df['round_trip_ok']]
smoke_passed = bool(ep_eligible['energyplus_ok'].fillna(False).infer_objects(copy=False).all())

print(f'\nSmoke test: {"PASSED" if smoke_passed else "FAILED"} ({len(smoke_df)} cases, '
      f'{rt_rejected} round-trip rejection(s))')
if rt_rejected:
    print(smoke_df[~smoke_df['round_trip_ok']][['tag']].to_string(index=False))
if not smoke_passed:
    print(ep_eligible[~ep_eligible['energyplus_ok'].fillna(False)].to_string())

Smoke sample: 8 cases

[1/8] ApartmentMidRise_1A_F4_llama-3.3-70b-versatile_seed1542799867 ... OK (271.0s)
    -> avg 271.1s/case, est. remaining 31.6 min
[2/8] ApartmentMidRise_1A_F4_gpt-oss-120b_seed1542799867 ... OK (308.1s)
    -> avg 289.6s/case, est. remaining 29.0 min
[3/8] ApartmentMidRise_7_F4_llama-3.3-70b-versatile_seed1542799867 ... OK (250.0s)
    -> avg 276.4s/case, est. remaining 23.0 min
[4/8] ApartmentMidRise_7_F4_gpt-oss-120b_seed1542799867 ... OK (308.2s)
    -> avg 284.3s/case, est. remaining 19.0 min
[5/8] OfficeMedium_1A_lobo_median_loo ... OK (82.7s)
    -> avg 244.0s/case, est. remaining 12.2 min
[6/8] OfficeMedium_1A_global_constant ... OK (80.8s)
    -> avg 216.8s/case, est. remaining 7.2 min
[7/8] OfficeMedium_7_lobo_median_loo ... OK (86.3s)
    -> avg 198.2s/case, est. remaining 3.3 min
[8/8] OfficeMedium_7_global_constant ... OK (86.7s)
    -> avg 184.2s/case, est. remaining 0.0 min

Smoke test: PASSED (8 cases, 0 round-trip rejection(s))


In [ ]:
# S4S-G  Full 1A/7 batch, parallel, checkpointed, one log file per climate
assert smoke_passed, 'Smoke test failed -- inspect smoke_test_results_stress.csv before proceeding.'

def _run_one_case(args):
    building, climate, source, fmt, model, seed, values, epw_path = args
    tag = f'{building}_{climate}_{source}'
    t0 = time.time()
    out_idf, (rt_passed, rt_report) = build_and_check(building, values, tag)
    if not rt_passed:
        return {'tag': tag, 'building': building, 'climate': climate, 'source': source,
                'format': fmt, 'model': model, 'seed': seed,
                'round_trip_ok': False, 'energyplus_ok': None,
                'duration_s': round(time.time() - t0, 1), 'rt_report': rt_report}
    ep_ok, _ = run_energyplus(out_idf, epw_path, f'{EP_OUT}/{tag}')
    return {'tag': tag, 'building': building, 'climate': climate, 'source': source,
            'format': fmt, 'model': model, 'seed': seed,
            'round_trip_ok': True, 'energyplus_ok': ep_ok,
            'duration_s': round(time.time() - t0, 1), 'rt_report': rt_report}


def _flatten_report(result):
    return [{'tag': result['tag'], 'building': result['building'], 'climate': result['climate'],
             'source': result['source'], 'format': result['format'], 'model': result['model'],
             'seed': result['seed'], 'duration_s': result['duration_s'], **p}
            for p in (result.get('rt_report') or [])]


def run_batch_parallel(manifest_df, epw_path, log_name, adherence_name):
    log_path = f'{SIM_DIR}/{log_name}'
    adherence_path = f'{SIM_DIR}/{adherence_name}'

    log = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
    adherence_rows = pd.read_csv(adherence_path).to_dict('records') if os.path.exists(adherence_path) else []
    done_tags = {r['tag'] for r in log if r.get('energyplus_ok') is not None or r.get('round_trip_ok') is False}

    jobs = []
    for _, row in manifest_df.iterrows():
        tag = f'{row["building"]}_{row["climate"]}_{row["source"]}'
        if tag in done_tags:
            continue
        values = {p: row[p] for p in TARGET_PARAMS}
        jobs.append((row['building'], row['climate'], row['source'],
                     row.get('format'), row.get('model'), row.get('seed'), values, epw_path))

    print(f'{len(jobs)} case(s) remaining (of {len(manifest_df)} total).')
    completed = 0
    batch_start = time.time()
    with ProcessPoolExecutor(max_workers=N_WORKERS) as executor:
        futures = {executor.submit(_run_one_case, job): job for job in jobs}
        for future in as_completed(futures):
            result = future.result()
            log.append({k: v for k, v in result.items() if k != 'rt_report'})
            adherence_rows.extend(_flatten_report(result))
            completed += 1
            status = ('OK' if result.get('energyplus_ok')
                       else 'ROUND-TRIP FAILED' if not result['round_trip_ok'] else 'EP FAILED')
            elapsed = time.time() - batch_start
            print(f'  [{completed}/{len(jobs)}] {result["tag"]}: {status} ({result["duration_s"]:.0f}s)  '
                  f'(avg {elapsed/completed:.0f}s/case, est. remaining {elapsed/completed*(len(jobs)-completed)/3600:.1f}h)')
            if completed % 10 == 0:
                pd.DataFrame(log).to_csv(log_path, index=False)
                pd.DataFrame(adherence_rows).to_csv(adherence_path, index=False)

    pd.DataFrame(log).to_csv(log_path, index=False)
    pd.DataFrame(adherence_rows).to_csv(adherence_path, index=False)
    return pd.DataFrame(log)

stress_batch_df = pd.concat([
    run_batch_parallel(stress_manifest_df[stress_manifest_df.climate == c], CLIMATE_EPW[c],
                        f'full_batch_log_{c}.csv', f'range_adherence_log_stress_{c}.csv')
    for c in CLIMATES['validation']
], ignore_index=True)
print(f'\n1A/7 stress test: {stress_batch_df.energyplus_ok.sum()}/{len(stress_batch_df)} successful.')
print(f'Mean duration: {stress_batch_df.duration_s.mean():.1f}s/case, '
      f'total: {stress_batch_df.duration_s.sum()/3600:.2f}h (sequential-equivalent)')

60 case(s) remaining (of 60 total).
  [1/60] ApartmentMidRise_1A_F4_llama-3.3-70b-versatile_seed1542799867: OK (507s)  (avg 507s/case, est. remaining 8.3h)
  [2/60] ApartmentMidRise_1A_F4_llama-3.3-70b-versatile_seed1542799868: OK (508s)  (avg 254s/case, est. remaining 4.1h)
  [3/60] ApartmentMidRise_1A_F4_llama-3.3-70b-versatile_seed1542799869: OK (478s)  (avg 328s/case, est. remaining 5.2h)
  [4/60] ApartmentMidRise_1A_F4_llama-3.3-70b-versatile_seed1542799870: OK (489s)  (avg 249s/case, est. remaining 3.9h)
  [5/60] ApartmentMidRise_1A_F4_llama-3.3-70b-versatile_seed1542799871: OK (494s)  (avg 296s/case, est. remaining 4.5h)
  [6/60] ApartmentMidRise_1A_F4_gpt-oss-120b_seed1542799867: OK (529s)  (avg 254s/case, est. remaining 3.8h)
  [7/60] ApartmentMidRise_1A_F4_gpt-oss-120b_seed1542799868: OK (525s)  (avg 286s/case, est. remaining 4.2h)
  [8/60] ApartmentMidRise_1A_F4_gpt-oss-120b_seed1542799869: OK (570s)  (avg 262s/case, est. remaining 3.8h)
  [9/60] ApartmentMidRise_1A_F4_gpt-o

In [ ]:
# S4S-H  Failure accounting
adherence_df = pd.concat([pd.read_csv(f'{SIM_DIR}/range_adherence_log_stress_{c}.csv') for c in CLIMATES['validation']],
                          ignore_index=True)
adherence_df.to_csv(f'{SIM_DIR}/range_adherence_log_stress.csv', index=False)
n_param_violations = (adherence_df['status'] != 'OK').sum()
print(f'{n_param_violations} parameter-level round-trip issue(s) across {len(adherence_df)} checked values.')
print(adherence_df[adherence_df['status'] != 'OK'].groupby(['model', 'climate', 'parameter', 'status']).size().to_string())

def build_failure_log(manifest_df, batch_df, climate_role):
    tags = manifest_df.apply(lambda r: f'{r["building"]}_{r["climate"]}_{r["source"]}', axis=1)
    merged = manifest_df.assign(tag=tags).merge(
        batch_df[['tag', 'round_trip_ok', 'energyplus_ok', 'duration_s']], on='tag', how='left')
    rows = []
    for _, r in merged.iterrows():
        if r['round_trip_ok'] is True and r['energyplus_ok'] is True:
            continue
        status = ('round_trip_failed' if r['round_trip_ok'] is False
                   else 'energyplus_failed' if r['energyplus_ok'] is False else 'not_yet_run')
        rows.append({'building': r['building'], 'climate': r['climate'], 'climate_role': climate_role,
                      'format': r.get('format'), 'model': r.get('model'), 'seed': r.get('seed'),
                      'source': r['source'], 'source_type': r['source_type'], 'status': status,
                      'duration_s': r.get('duration_s')})
    return pd.DataFrame(rows)

stress_failures = build_failure_log(stress_manifest_df, stress_batch_df, 'weather_file_stress_test')
stress_failures.to_csv(f'{SIM_DIR}/simulation_failure_log_stress.csv', index=False)
print(f'\n{len(stress_failures)} non-successful vector(s) out of {len(stress_manifest_df)}.')
if len(stress_failures):
    print(stress_failures.groupby(['status', 'model', 'climate']).size().to_string())

0 parameter-level round-trip issue(s) across 1080 checked values.
Series([], )

0 non-successful vector(s) out of 120.


In [ ]:
# S4S-I  Final validation + environment record
rate = stress_batch_df.energyplus_ok.mean() if len(stress_batch_df) else 0
print(f'1A/7 stress: {stress_batch_df.energyplus_ok.sum()}/{len(stress_batch_df)} ({rate*100:.1f}%)')

with open(f'{DATA}/environment_sprint4_stress.json', 'w') as f:
    json.dump({'sprint': 'Sprint 4 - Weather-File Stress Test', 'python_version': sys.version,
               'platform': platform.platform(), 'energyplus_version': energyplus_version,
               'n_workers': N_WORKERS, 'ep_timeout_s': EP_TIMEOUT_S,
               'master_seed': protocol['master_seed']}, f, indent=2)
print('Saved: environment_sprint4_stress.json')

1A/7 stress: 120/120 (100.0%)
Saved: environment_sprint4_stress.json


## Sprint 4 — Weather-File Stress Test (S4S) — Overview / Genel Bakış

**EN — What this notebook does and why.** Runs the same 9-parameter
injection → round-trip verification → EnergyPlus simulation pipeline as
the main experiment (S4M), but for two additional weather files (Miami/1A,
International Falls/7) applied to the *same* Buffalo-derived reference
IDFs. This is deliberately **not** an independent multi-climate DOE
prototype validation — no climate-specific prototype IDF exists for 1A/7
in this study — so it is scoped and reported separately as a robustness
check: does an F4 (fully-constrained) parameter set, generated once under
Buffalo/5A context, still produce a physically coherent simulation when
run against a hot-humid and a very-cold weather file? Only F4 (the format
that combines unit description + numeric range) and the two deterministic
baselines are tested here — not F1/F2/F3 and not uniform-random draws,
since the goal is a targeted robustness check, not a full replica of the
main experiment.

**TR — Bu notebook ne yapıyor ve neden.** Ana deneyle (S4M) birebir aynı
9-parametre enjeksiyon → round-trip doğrulama → EnergyPlus simülasyonu
hattını, *aynı* Buffalo-türetilmiş referans IDF'lere uygulanan iki ek hava
dosyası (Miami/1A, International Falls/7) için çalıştırır. Bu, bilinçli
olarak **bağımsız bir çoklu-iklim DOE prototip doğrulaması değildir** — bu
çalışmada 1A/7 için iklime özgü bir prototip IDF yoktur — bu yüzden ayrı,
kapsamı sınırlı bir sağlamlık kontrolü olarak raporlanır: Buffalo/5A
bağlamında bir kez üretilen bir F4 (tam kısıtlı) parametre kümesi, sıcak-
nemli ve çok-soğuk bir hava dosyasına karşı çalıştırıldığında hâlâ
fiziksel olarak tutarlı bir simülasyon üretiyor mu? Yalnızca F4 (birim
açıklaması + sayısal aralığı birleştiren format) ve iki deterministik
baseline test edilir — F1/F2/F3 ve uniform-rastgele çekilişler değil,
çünkü amaç ana deneyin tam bir kopyası değil, hedefli bir sağlamlık
kontrolüdür.

### Blok blok / Block by block

| Blok | EN | TR |
|---|---|---|
| **S4S-0** | Setup: mounts Drive, installs/verifies EnergyPlus 22.1.0, loads protocol/LOBO/control files, sets `CLIMATE_EPW={1A, 7}` and per-parameter tolerances. | Kurulum: Drive'ı bağlar, EnergyPlus 22.1.0'ı kurar/doğrular, protokol/LOBO/kontrol dosyalarını yükler, `CLIMATE_EPW={1A, 7}` ve parametre toleranslarını ayarlar. |
| **S4S-A** | Comment-anchored parameter extraction — identical logic to S4M-A. | Yorum-çapalı parametre çıkarımı — S4M-A ile birebir aynı mantık. |
| **S4S-B** | Injection — identical logic to S4M-B. | Enjeksiyon — S4M-B ile birebir aynı mantık. |
| **S4S-C** | Adds standard `Output:Meter`/`Output:Variable` objects to every IDF. | Her IDF'ye standart `Output:Meter`/`Output:Variable` nesneleri ekler. |
| **S4S-D** | Round-trip check (write → read back → compare) + EnergyPlus subprocess runner with timeout handling. | Round-trip kontrolü (yaz → geri oku → karşılaştır) + zaman aşımı yönetimli EnergyPlus çalıştırıcı. |
| **S4S-E** | Builds the 120-vector manifest: 100 F4-only LLM outputs (5 buildings × 2 models × 5 seeds × 2 climates) + 20 deterministic baselines (5 buildings × 2 baseline types × 2 climates). | 120 vektörlük manifesti üretir: 100 yalnızca-F4 LLM çıktısı (5 bina × 2 model × 5 tohum × 2 iklim) + 20 deterministik baseline (5 bina × 2 baseline türü × 2 iklim). |
| **S4S-F** | Smoke test: one case per (model, climate) + one per (baseline, climate) — 8 cases, mandatory gate before the full batch. | Smoke test: (model, iklim) başına bir vaka + (baseline, iklim) başına bir vaka — 8 vaka, tam batch öncesi zorunlu kapı. |
| **S4S-G** | Full batch: runs all 120 vectors in parallel, per climate, checkpointed every 10 completions. | Tam batch: 120 vektörün tamamını iklim başına paralel çalıştırır, her 10 tamamlanmada bir checkpoint. |
| **S4S-H** | Failure accounting: merges both climates' adherence logs, summarizes any parameter-level or simulation-level failures. | Hata dökümü: iki iklimin adherence log'unu birleştirir, parametre veya simülasyon düzeyindeki tüm hataları özetler. |
| **S4S-I** | Final validation + writes the environment record (EnergyPlus version, worker count, timeout, seed). | Son doğrulama + ortam kaydını yazar (EnergyPlus sürümü, işçi sayısı, zaman aşımı, tohum). |

### Çıktı dosyaları / Output files

| Dosya / File | İçerik / Contents |
|---|---|
| `simulation_vector_manifest_weather_stress.csv` | 120 vektörün tamamı / all 120 vectors |
| `smoke_test_results_stress.csv` | 8 vakalık pilot koşu / 8-case pilot run |
| `full_batch_log_1A.csv`, `full_batch_log_7.csv` | İklim başına vektör sonucu + süre / per-climate outcome + duration |
| `range_adherence_log_stress_1A.csv`, `_7.csv`, birleşik `range_adherence_log_stress.csv` | Parametre başına hedef/geri-okuma/fark/durum / per-parameter target/readback/diff/status |
| `simulation_failure_log_stress.csv` | Başarısız her vektör (varsa) / any non-successful vector |
| `environment_sprint4_stress.json` | EnergyPlus sürümü, ayarlar, master seed / EnergyPlus version, settings, master seed |
| `idf_llm/*_1A_*.idf`, `*_7_*.idf` | Üretilen IDF'ler (ana deneyle paylaşılan klasör) / generated IDFs (shared folder with main experiment) |
| `sonuclar/<tag>/` | Ham EnergyPlus çıktısı, vektör başına / raw EnergyPlus output, per vector |

### Sonuç / Result
**120/120 (%100) başarılı, 0 round-trip veya simülasyon hatası.**
**120/120 (100%) successful, 0 round-trip or simulation failures.**